# Misconception Extraction and Prompt Comparison

Extracts the per-turn misconception channel (present / absent / not_evidenced)
from MathDial dialogues via the OpenRouter API, and compares prompt variants on
two columns:

- **Endpoint agreement** (automatic screen): does the last substantive label
  agree with the dialogue's resolution field? Runs over many dialogues, no
  human labelling. A screen for broken prompts, not the final selector.
- **Human agreement** (selector): does the prompt match the author's labels on
  the frozen validation set, at trinary and binary granularity? This is what
  actually picks the prompt.

All logic lives in `extension/myext`; this notebook orchestrates. Prompts live
as files in `extension/myext/prompts/` and are compared by filename.

**Before running:** export your key in the terminal that launches Jupyter:
`export OPENROUTER_API_KEY=sk-or-...`  then restart the kernel.


## Setup

In [1]:
import os
from pathlib import Path

_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "data" / "annotated").exists():
        os.chdir(_c); break
else:
    raise FileNotFoundError("Could not find data/annotated above " + str(_here))

import sys
sys.path.insert(0, str(Path.cwd() / "extension"))
print("working directory:", os.getcwd())
print("API key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


working directory: /Users/tandon.utsav2/Desktop/Experiment_1
API key set: True


In [2]:
from ast import literal_eval
import pandas as pd

from myext import (filtering, splits, prompt_loader, extraction,
                   endpoint, agreement, validation_set)

DATA = Path("data/annotated")
CONV = {c: literal_eval for c in ["annotation", "dialogue"]}

def load(split):
    df = pd.read_csv(DATA / f"mathdial_{split}_atc.csv", converters=CONV)
    df, n = filtering.drop_failed_annotations(df)
    df.index = df["index"]            # index by the dialogue id used everywhere
    print(f"{split}: {len(df)} dialogues ({n} failed-annotation removed)")
    return df

train = load("train")


train: 2235 dialogues (18 failed-annotation removed)


## Frozen split

Splits the training dialogues once into dev / validation / pool, seeded and
cached to disk. Developed prompts on **dev**; the **validation** set is held
out for human labelling and prompt selection; **pool** is for the eventual
full run. The split never changes once written.


In [3]:
sp = splits.make_or_load_splits(list(train.index), n_dev=8, n_validation=55, seed=7)
print(sp.summary())


dev=8  validation=55  pool=2172  total=2235


## Export the validation set for labelling

Writes one row per student turn in the validation dialogues, with the problem,
the misconception in plain language, the dialogue history, and an empty
`label` column. **Open this CSV and fill the `label` column** with
`present` / `absent` / `not_evidenced` from the codebook (Appendix of the
report). You can label it gradually; the comparison below scores whatever is
labelled so far.


In [4]:
VAL_CSV = "extension/artifacts/validation_to_label.csv"
if not Path(VAL_CSV).exists():
    validation_set.export_validation_csv(train, sp.validation, VAL_CSV)
    print("wrote", VAL_CSV, "- open and fill the 'label' column")
else:
    print(VAL_CSV, "already exists (not overwriting your labels)")


extension/artifacts/validation_to_label.csv already exists (not overwriting your labels)


## Step 1: develop a prompt on the dev set

Run one prompt over the small dev set and read the labels by eye against the
codebook. This is where you iterate on prompt wording. Cheap: 8 dialogues.
Cached, so re-running is free.


In [7]:
DEV_PROMPT = "codebook_detailed"   # change to try a different prompt file
tmpl = prompt_loader.load_prompt(DEV_PROMPT)

dev_results = extraction.extract_many(train, sp.dev, tmpl, DEV_PROMPT)

# print the labels for eyeballing
for did in sp.dev[:3]:
    print(f"--- dialogue {did} | resolution={train.loc[did,'self-correctness']} ---")
    print("    misconception:", str(train.loc[did,'teacher_described_confusion'])[:90])
    labs = dev_results[did]
    if "_error" in labs:
        print("    ERROR:", labs["_error"]); continue
    for k in sorted(labs, key=lambda x: int(''.join(c for c in x if c.isdigit()) or 0)):
        print(f"    {k}: {labs[k]['label']:14s} | {labs[k]['reason']}")
    print()


  [codebook_detailed] 8/8 done (0 errors)
--- dialogue 278 | resolution=Yes ---
    misconception: Student misread the question and thought it was asking for total gallons but is asking for
    turn 1: absent         | correctly identifies the 2 more gallons, engaging the more-vs-total reasoning the misconception corrupts and getting the relevant quantity right
    turn 2: present        | gives the total (5 gallons) as the final answer rather than the requested 'more' amount, an instance of confusing which quantity is relevant
    turn 3: present        | still frames the answer around the total in total even when asked specifically for 'more', relying on the faulty relevance reasoning

--- dialogue 519 | resolution=No ---
    misconception: overcomplicated things
    turn 1: present        | overcomplicates with two parallel calculations and adds a nonexistent empty-bag weight, misidentifying the problem structure
    turn 2: present        | repeats the same overcomplicated structur

## Step 2: run all prompts over the validation set

Runs every prompt in `myext/prompts/` over the validation dialogues, caching
each (dialogue, prompt) result. This produces the labels both the endpoint
screen and the human-agreement selector are computed from.


In [6]:
all_prompts = prompt_loader.list_prompts()
print("prompts to compare:", all_prompts)

val_results = {}
for name in all_prompts:
    t = prompt_loader.load_prompt(name)
    print(f"running {name} over {len(sp.validation)} validation dialogues...")
    val_results[name] = extraction.extract_many(train, sp.validation, t, name)


prompts to compare: ['codebook_concise', 'codebook_concise_familyonly', 'codebook_detailed', 'codebook_detailed_familyonly', 'few_shot', 'few_shot_familyonly', 'minimal', 'minimal_familyonly', 'negative_guard', 'negative_guard_familyonly', 'opportunity_framing', 'opportunity_framing_familyonly', 'role_expert', 'role_expert_familyonly', 'stepwise_define', 'stepwise_define_familyonly']
running codebook_concise over 55 validation dialogues...


KeyboardInterrupt: 

## Step 3: the comparison table

One row per prompt. Endpoint agreement and label distribution are automatic
(available now). Human agreement fills in from whatever you have labelled in
the validation CSV so far (raw and chance-corrected kappa, trinary and binary).

Read across a row: is the prompt coherent (endpoint), non-degenerate
(distribution), and accurate against your labels (human agreement)? Read down:
which prompt wins. Endpoint **screens**, human agreement **selects**.


In [ ]:
# load whatever human labels exist so far (safe if partially labelled or empty)
try:
    human = validation_set.load_validation_labels(VAL_CSV)
    print(f"loaded {len(human)} human-labelled turns")
except Exception as e:
    human = {}
    print("no human labels yet (or file unfilled):", e)

rows = []
for name, res in val_results.items():
    ep = endpoint.endpoint_check(res, train)
    ag = agreement.agreement(res, human) if human else None
    row = {
        "prompt": name,
        "endpoint_agree": round(ep.agreement, 3),
        "excluded": ep.n_excluded_no_substantive,
        "dist_present": round(ep.label_distribution["present"], 2),
        "dist_absent": round(ep.label_distribution["absent"], 2),
        "dist_notev": round(ep.label_distribution["not_evidenced"], 2),
    }
    if ag and ag.n_turns:
        row.update({
            "human_n": ag.n_turns,
            "kappa_trinary": round(ag.kappa_trinary, 3),
            "kappa_binary": round(ag.kappa_binary, 3),
        })
    rows.append(row)

table = pd.DataFrame(rows)
print(table.to_string(index=False))


### Reading the table

- **endpoint_agree**: screen. Low values flag a broken prompt. Do not select
  on this alone; it only checks endings and the resolution field is imperfect.
- **dist_***: a near-degenerate distribution (one class dominating) is a red
  flag even with a decent endpoint score.
- **kappa_trinary / kappa_binary**: the selector, once enough turns are
  labelled. Chance-corrected agreement with the author labels. Highest kappa
  is the prompt to choose. Trinary is the harder, finer distinction.
- Compare the endpoint column against the kappa columns across prompts: if they
  rank prompts the same way, the cheap screen is trustworthy; if they diverge,
  trust the human kappa for selection.

Once a prompt is chosen, freeze it and run it over the full pool + test for the
experiment (a later notebook).
